# Oscillating Droplet Analysis

Verification of the CSF surface tension implementation via an oscillating droplet.

Setup (same physics as spurious currents):
- Initially **elliptical** drop: r(θ) = R₀(1 + ε·cos(2θ)), R₀=0.2, ε=0.1
- 1×1 box, symmetry BCs, quasi-2D (one element in z)
- ρ=300, μ=0.1, σ=1.0 → La = ρσD/μ² = **12000**

Expected behavior: the drop oscillates as surface tension restores it to a
circle, with kinetic energy showing periodic oscillation and viscous decay.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as tri
from mpi4py import MPI

from pysemtools.io.ppymech.neksuite import preadnek
from pysemtools.datatypes.msh import Mesh as msh_c
from pysemtools.datatypes.coef import Coef as coef_c
from pysemtools.datatypes.field import Field as field_c

comm = MPI.COMM_WORLD

# Physics parameters
mu = 0.1
sigma = 1.0
rho = 300.0
R0 = 0.2
D = 2 * R0
La = rho * sigma * D / mu**2
print(f"La = {La:.0f}")

# Discover field files
import glob, os
field_files = sorted(glob.glob("field0.f[0-9]*"))
N_SNAPSHOTS = len(field_files)
print(f"Found {N_SNAPSHOTS} field files")

## Load mesh and build triangulation

In [ ]:
xyz_info = preadnek(field_files[0], comm)
msh = msh_c(comm, data=xyz_info)
coef = coef_c(msh, comm)

# Build element-local triangulation from the structured GLL grid.
n_elem = msh.x.shape[0]
lx = msh.x.shape[3]
ly = msh.x.shape[2]

x_all, y_all, triangles = [], [], []
offset = 0
for e in range(n_elem):
    x_all.append(msh.x[e, 0, :, :].flatten())
    y_all.append(msh.y[e, 0, :, :].flatten())
    for j in range(ly - 1):
        for i in range(lx - 1):
            p0 = offset + j * lx + i
            p1 = offset + j * lx + (i + 1)
            p2 = offset + (j + 1) * lx + i
            p3 = offset + (j + 1) * lx + (i + 1)
            triangles.append([p0, p1, p3])
            triangles.append([p0, p3, p2])
    offset += lx * ly

x = np.concatenate(x_all)
y = np.concatenate(y_all)
triang = tri.Triangulation(x, y, np.array(triangles))
print(f"{n_elem} elements, {len(x)} points, {len(triangles)} triangles")

## Part A: Flow field visualization at multiple times

Shows the drop shape and velocity field at several snapshots through the
oscillation cycle.

In [ ]:
# Select ~6 evenly spaced snapshots to show the oscillation
if N_SNAPSHOTS >= 6:
    snap_indices = np.linspace(0, N_SNAPSHOTS - 1, 6, dtype=int)
else:
    snap_indices = np.arange(N_SNAPSHOTS)

n_plots = len(snap_indices)
fig, axes = plt.subplots(2, n_plots, figsize=(4 * n_plots, 8), dpi=150)
if n_plots == 1:
    axes = axes[:, np.newaxis]

for col, idx in enumerate(snap_indices):
    data = preadnek(field_files[idx], comm)
    fld = field_c(comm, data=data)
    t = fld.t

    phi = fld.fields["scal"][0][:, 0, :, :].flatten()
    u = fld.fields["vel"][0][:, 0, :, :].flatten()
    v = fld.fields["vel"][1][:, 0, :, :].flatten()
    vel_mag = np.sqrt(u**2 + v**2)

    # Row 1: Phase field
    ax = axes[0, col]
    c = ax.tricontourf(triang, phi, levels=100, cmap="RdBu_r", vmin=0, vmax=1)
    ax.tricontour(triang, phi, levels=[0.5], colors="w", linewidths=1.2)
    ax.set_aspect("equal")
    ax.set_title(f"t = {t:.3f}", fontsize=10)
    if col == 0:
        ax.set_ylabel(r"$\phi$")

    # Row 2: Velocity magnitude
    ax = axes[1, col]
    c = ax.tricontourf(triang, vel_mag, levels=100, cmap="viridis")
    ax.tricontour(triang, phi, levels=[0.5], colors="w", linewidths=1.2,
                  linestyles="--")
    ax.set_aspect("equal")
    if col == 0:
        ax.set_ylabel(r"$|\mathbf{u}|$")

fig.suptitle(
    f"Oscillating droplet — La={La:.0f}, $\\sigma$={sigma}, $\\mu$={mu},"
    f" $\\epsilon$=0.1",
    y=1.02)
plt.tight_layout()
plt.savefig("oscillating_droplet_snapshots.png", dpi=150, bbox_inches="tight")
plt.show()

## Part B: Kinetic energy vs time

The kinetic energy Ekin(t) should show periodic oscillation (surface tension
drives the drop shape back and forth) with viscous decay (the oscillation
amplitude decreases over time as energy is dissipated).

This is the key verification: the CSF method should produce physically
correct oscillatory behavior, not just static equilibrium.

In [ ]:
# Load ekin.csv (columns: t, Ekin, enst, u_max)
ekin_csv = np.genfromtxt("ekin.csv", delimiter=",", comments="#")
t_csv = ekin_csv[:, 0]
Ekin_csv = ekin_csv[:, 1]
enst_csv = ekin_csv[:, 2]
u_max_csv = ekin_csv[:, 3]

# Non-dimensional time: t* = sigma * t / (mu * D)
t_star = sigma * t_csv / (mu * D)

fig, axes = plt.subplots(1, 3, figsize=(15, 4), dpi=150)

# Kinetic energy
ax = axes[0]
ax.plot(t_star, Ekin_csv, "-", lw=1)
ax.set_xlabel(r"$t^* = \sigma t / (\mu D)$")
ax.set_ylabel(r"$E_{kin}$")
ax.set_title("Kinetic energy")
ax.grid(True)

# u_max (shows oscillation amplitude)
ax = axes[1]
ax.plot(t_star, u_max_csv, "-", lw=1)
ax.set_xlabel(r"$t^* = \sigma t / (\mu D)$")
ax.set_ylabel(r"$u_{\max}$")
ax.set_title("Maximum velocity")
ax.grid(True)

# Enstrophy
ax = axes[2]
ax.plot(t_star, enst_csv, "-", lw=1)
ax.set_xlabel(r"$t^* = \sigma t / (\mu D)$")
ax.set_ylabel(r"$\Omega$")
ax.set_title("Enstrophy")
ax.grid(True)

plt.suptitle(
    f"Oscillating droplet — La={La:.0f}, $\\sigma$={sigma}",
    y=1.02)
plt.tight_layout()
plt.savefig("oscillating_droplet_ekin.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Peak Ekin: {Ekin_csv.max():.4e}")
print(f"Final Ekin: {Ekin_csv[-1]:.4e}")
print(f"Peak u_max: {u_max_csv.max():.4e}")

## Part C: Cross-check Ekin from field files vs ekin.csv

In [ ]:
V = float(np.sum(coef.B))
times_field = []
Ekin_fields = []

for fname in field_files:
    data = preadnek(fname, comm)
    fld = field_c(comm, data=data)
    u_snap = fld.fields["vel"][0]
    v_snap = fld.fields["vel"][1]
    w_snap = fld.fields["vel"][2] if len(fld.fields["vel"]) > 2 else np.zeros_like(u_snap)
    uu = u_snap**2 + v_snap**2 + w_snap**2
    times_field.append(fld.t)
    Ekin_fields.append(0.5 * float(np.sum(uu * coef.B)) / V)

times_field = np.array(times_field)
Ekin_fields = np.array(Ekin_fields)

fig, ax = plt.subplots(figsize=(7, 4), dpi=150)
ax.plot(times_field, Ekin_fields, "o-", label="from field files", markersize=3)
ax.plot(t_csv, Ekin_csv, "-", alpha=0.7, label="ekin.csv")
ax.set_xlabel("time $t$")
ax.set_ylabel("kinetic energy $E_{kin}$")
ax.set_title("Ekin cross-check: field files vs ekin.csv")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.savefig("Ekin_crosscheck.png", dpi=150, bbox_inches="tight")
plt.show()

# Print comparison at matching times
print("Time     Ekin (fields)    Ekin (csv)       rel. diff")
for t_f, e_f in zip(times_field, Ekin_fields):
    idx = np.argmin(np.abs(t_csv - t_f))
    e_c = Ekin_csv[idx]
    if e_c > 0:
        rel = abs(e_f - e_c) / e_c
        print(f"{t_f:.4f}   {e_f:.6e}   {e_c:.6e}   {rel:.2e}")
    else:
        print(f"{t_f:.4f}   {e_f:.6e}   {e_c:.6e}   (zero)")